In [1]:
!git clone https://github.com/sara-graca/acoustic-vs-neural-representations.git
%cd acoustic-vs-neural-representations
!pip install tgt chardet

Cloning into 'acoustic-vs-neural-representations'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 35 (delta 6), reused 32 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 32.97 KiB | 1.43 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/kaggle/working/acoustic-vs-neural-representations
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 1.7 MB/s eta 0:00:00


In [2]:
import shutil
import os

os.makedirs("data/parsed", exist_ok=True)
os.makedirs("data/features", exist_ok=True)
os.makedirs("data/raw", exist_ok=True)

shutil.copy(
    "/kaggle/input/datasets/sara2graca/statistics-parsed-data/phonemes.csv",
    "data/parsed/phonemes.csv"
)

shutil.copytree(
    "/kaggle/input/datasets/sara2graca/statistics-all-data/ru-fr_interference",
    "data/raw/ru-fr_interference"
)

print("Data ready")

Data ready


In [3]:
import os
import yaml
import numpy as np
import pandas as pd
import torch
from transformers import WhisperProcessor, WhisperModel
from tqdm.notebook import tqdm
import librosa

MODEL_NAME = "openai/whisper-medium"
LAYER      = 20
INPUT      = "data/parsed/phonemes.csv"
OUTPUT     = "data/features/features_whisper.npz"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using: {device}")

processor = WhisperProcessor.from_pretrained(MODEL_NAME)
model     = WhisperModel.from_pretrained(MODEL_NAME, output_hidden_states=True)
model     = model.to(device)
model.eval()

df = pd.read_csv(INPUT)

# fix wav paths to point to kaggle data
df["wav_path"] = df["wav_path"].apply(
    lambda p: os.path.join(
        "/kaggle/input/datasets/sara2graca/statistics-all-data",
        p.replace("data\\raw\\", "").replace("data/raw/", "").replace("\\", "/")
    )
)

embeddings = {}
current_wav, current_audio, current_sr = None, None, None

for i, row in tqdm(df.iterrows(), total=len(df)):
    if row["wav_path"] != current_wav:
        current_wav = row["wav_path"]
        try:
            current_audio, current_sr = librosa.load(current_wav, sr=16000)
        except Exception:
            current_audio, current_sr = None, None
    
        if current_audio is None:
            embeddings[i] = np.zeros(model.config.d_model)
            continue

    onset_sample  = int(row["onset"] * current_sr)
    offset_sample = int(row["offset"] * current_sr)
    segment = current_audio[onset_sample:offset_sample]

    if len(segment) < 10:
        embeddings[i] = np.zeros(model.config.d_model)
        continue

    inputs = processor(segment, sampling_rate=current_sr, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.encoder(inputs.input_features, output_hidden_states=True)

    hidden   = outputs.hidden_states[LAYER]
    seq_len  = hidden.shape[1]

    total_frames = inputs.input_features.shape[-1] // 2
    start_frame  = min(int(row["onset"] * total_frames / 30.0), seq_len - 1)
    dur_frames   = max(1, int((row["offset"] - row["onset"]) * total_frames / 30.0))
    end_frame    = min(start_frame + dur_frames, seq_len)

    embedding     = hidden[0, start_frame:end_frame, :].mean(dim=0).cpu().numpy()
    embeddings[i] = embedding

os.makedirs(os.path.dirname(OUTPUT), exist_ok=True)
np.savez(OUTPUT, **{str(k): v for k, v in embeddings.items()})
print(f"Done: {len(embeddings)} embeddings → {OUTPUT}")

Using: cuda


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.06G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

  0%|          | 0/22919 [00:00<?, ?it/s]

Done: 22919 embeddings → data/features/features_whisper.npz
